# 1x2 MMI Splitter — EME

Simulate a 1x2 MMI splitter with the BEEF library using Eigenmode Expansion (EME).

**Geometry:**
- Single-mode input waveguide -> MMI body -> two output waveguides
- SOI platform: 160 nm Si on SiO2
- Dimensions from typical fabricated device (mm_length = 4.5 um)

**Method:** EME via meow. Fast, no FDTD required. Models the path from the input to each
output waveguide in turn (one solve per output_idx — see the wavelength-sweep cell) and
reports both. For a balanced 1x2 splitter each output carries ~50% of the input power
(ideal insertion loss: -3 dB per port).

In [ ]:
import os
import numpy as np
import matplotlib

from beef import MMI, EMESolver, SOI_160, plot_eme_geometry, plot_performance

# Output to a top-level example_output/ folder outside the repo
_HERE = os.path.dirname(os.path.abspath(__file__))
output_dir = os.path.abspath(os.path.join(_HERE, "..", "..", "..", "..", "example_output", "mmi_1x2", "mmi_1x2_eme"))
save_dir = os.path.abspath(os.path.join(_HERE, "..", "..", "..", "..", "example_sim_cache"))

In [ ]:
# ---------------------------------------------------------------------------
# Global simulation parameters
# ---------------------------------------------------------------------------
RESOLUTION = 40.0   # mesh resolution (pixels/µm) — used by the EME solver
WL_MIN = 1.26       # wavelength sweep start (µm)
WL_MAX = 1.36       # wavelength sweep end (µm)
N_WL = 11            # number of wavelength points

In [ ]:
# ---------------------------------------------------------------------------
# 1. Define geometry
# ---------------------------------------------------------------------------
mmi = MMI(
    n_inputs=1,             # 1x2 splitter
    n_outputs=2,
    mm_length=4.5,          # MMI body length (µm)
    mm_width=2.0,           # MMI body width (µm)
    waveguide_width=0.35,   # single-mode access waveguide width (µm)
    taper_length=1.0,       # input/output taper length (µm)
    taper_width=0.8,        # taper width at MMI junction (µm)
    waveguide_offset=1.0,   # center-to-center output spacing (µm)
    waveguide_height=0.16,  # Si layer thickness (µm) — SOI_160
)

print(f"MMI total length: {mmi.total_length:.2f} µm")

In [ ]:
# ---------------------------------------------------------------------------
# 2. Configure EME solver
# ---------------------------------------------------------------------------
solver = EMESolver(
    # 5 regions: wg_in | taper_in | MMI body | taper_out | wg_out
    num_modes=[1,   5,   12,  5,   1  ],  # modes per region
    num_cells=[1,   8,  1,   8,  1  ],  # cells per region (tapers subdivided)
    lengths=  [0.5, 1.0, 4.5, 1.0, 0.5], # region lengths (µm)
    oxide_width=5.0,                       # cross-section mesh y-span (µm)
    oxide_height=1.5,                      # cross-section mesh z-span (µm)
    res=RESOLUTION,                        # mesh pts/µm
    material_stack=SOI_160,                # 160 nm SOI stack
    mode_solver="tidy3d",                  # tidy3d-backed mode solver (no FDTD section to stay consistent with)
)

In [ ]:
# ---------------------------------------------------------------------------
# 3. Geometry plot
# ---------------------------------------------------------------------------
os.makedirs(output_dir, exist_ok=True)
plot_eme_geometry(
    mmi, solver=solver,
    region_labels=["wg in", "taper in", "MMI body", "taper out", "wg out"],
    title="1×2 MMI Splitter — Top-down Geometry",
    output_path=os.path.join(output_dir, "geometry.png"),
    # output_idx omitted (defaults to None) — draws both output branches,
    # since both are solved below (each solve() call restricts them
    # one at a time; the plot shows the full device for reference).
)
print(f"Geometry plot saved → {output_dir}/geometry.png")

In [ ]:
# ---------------------------------------------------------------------------
# 4. Wavelength sweep
# ---------------------------------------------------------------------------
wavelengths = np.linspace(WL_MIN, WL_MAX, N_WL)  # µm, O-band

print(f"Sweeping {len(wavelengths)} wavelengths: "
      f"{wavelengths[0]:.3f} – {wavelengths[-1]:.3f} µm")

# solve()'s single **kwargs call applies the same output_idx to every entry
# in `ports`, so measuring both outputs accurately takes two separate calls
# — one per output_idx — each restricting the EME cross-sections to that
# one output branch only (output_ys[idx]), giving the isolated waveguide
# mode rather than a supermode of both outputs (the two branches are
# `waveguide_offset - waveguide_width` = 0.65 µm apart edge-to-edge past the
# taper, comfortably decoupled). Results are merged into one SMatrix, same
# as mmi_1x2_fdtd.py/mmi_1x2_hybrid.py's two-output reporting.
smat_top = solver.solve(mmi, wavelengths, ports=["out_top"], output_idx=1, save_dir=save_dir)
smat_bot = solver.solve(mmi, wavelengths, ports=["out_bot"], output_idx=0, save_dir=save_dir)
smat = smat_top
smat._data.update(smat_bot._data)
smat.check_passive()  # sanity: total output power must not exceed 1 at any wavelength

In [ ]:
# ---------------------------------------------------------------------------
# 5. Print summary
# ---------------------------------------------------------------------------
T_top, T_bot = smat.transmission("out_top"), smat.transmission("out_bot")
T_total = T_top + T_bot
wl_center = wavelengths[len(wavelengths) // 2]
idx = np.argmin(np.abs(wavelengths - wl_center))

print(f"\nResults at λ = {wl_center:.3f} µm:")
print(f"  Top output    T = {T_top[idx]:.4f}  IL = {10*np.log10(max(T_top[idx],1e-20)):.2f} dB")
print(f"  Bottom output T = {T_bot[idx]:.4f}  IL = {10*np.log10(max(T_bot[idx],1e-20)):.2f} dB")
print(f"  Total         T = {T_total[idx]:.4f}  IL = {10*np.log10(max(T_total[idx],1e-20)):.2f} dB")
print(f"  (ideal 50/50 split → T_each=0.5, IL_total=0.0 dB)")

In [ ]:
# ---------------------------------------------------------------------------
# 6. Save results
# ---------------------------------------------------------------------------

smat.save(os.path.join(output_dir, "smatrix.pkl"))
print(f"\nSMatrix saved → {output_dir}/smatrix.pkl")

# Plot insertion loss and transmission vs wavelength — plot_performance()
# defaults to every port in smat, so both outputs show up automatically.
plot_performance(smat, output_dir=output_dir, title="1×2 MMI Splitter (EME)")
print(f"Performance plot saved → {output_dir}/performance.png")